# Structured Chain-of-Thought (S-CoT) SFT Training
This notebook performs Supervised Fine-Tuning (SFT) on the **Qwen2.5-1.5B-Instruct** model using specialized reasoning traces. It is optimized for the **Google Colab Free Tier (T4 GPU)** using the [Unsloth](https://github.com/unslothai/unsloth) library.

### 1. Environment Setup
We use `uv` for ultra-fast dependency resolution to avoid the "infinite loop" sometimes seen with standard `pip` in Colab.

In [ ]:
%%capture
# Install uv for fast dependency resolution
!pip install uv

# Install Unsloth and its friends using uv (much faster and avoids backtrack hangs)
!uv pip install --system "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install --system "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes jsonlines

### 2. Clone Repository & Load Data
Cloning the `scot-reasoning-` repo to access the dataset.

In [ ]:
import os
REPO_URL = "https://github.com/TinevimboMusingadi/scot-reasoning-.git"
REPO_DIR = "scot-reasoning-"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

DATA_PATH = os.path.join(REPO_DIR, "data/full_run/scot_traces.jsonl")
print(f"Data path: {DATA_PATH}")

### 3. Initialize Model and Tokenizer
We load the model using 4-bit quantization to fit in 16GB VRAM, and add the special S-CoT tokens.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_len = 2048
dtype = None # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_len,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

SCOT_TOKENS = [
    "<reasoning>", "</reasoning>",
    "<meta_reasoning>", "</meta_reasoning>",
    "<abduction>", "</abduction>",
    "<decompose>", "</decompose>",
    "<deduction>", "</deduction>",
    "<induction>", "</induction>",
    "<analogy>", "</analogy>",
    "<causal>", "</causal>",
    "<answer>", "</answer>",
]

tokenizer.add_special_tokens({"additional_special_tokens": SCOT_TOKENS})
model.resize_token_embeddings(len(tokenizer))

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 4. Data Preparation
Formatting the JSONL data into the ChatML-style prompt used by the student model.

In [ ]:
import jsonlines
from datasets import Dataset

def formatting_prompts_func(examples):
    problems = examples["problem"]
    traces   = examples["scot_trace"]
    texts = []
    for problem, trace in zip(problems, traces):
        # Match the build_prompt logic from training/sft_scot.py
        text = f"<|im_start|>user\n{problem}<|im_end|>\n<|im_start|>assistant\n{trace}<|im_end|>"
        texts.append(text)
    return { "text" : texts, }

data_list = []
with jsonlines.open(DATA_PATH) as reader:
    for obj in reader:
        data_list.append(obj)

dataset = Dataset.from_list(data_list)
dataset = dataset.map(formatting_prompts_func, batched = True)

### 5. Training
Running the optimized SFT trainer.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_len,
    dataset_num_proc = 2,
    args = TrainingArguments(
        num_train_epochs = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        max_steps = 750, # Set to a higher value like 500 for full training
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 6. Inference / Testing
Test the model on a new problem to check if it follows the structured reasoning format.

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    f"<|im_start|>user\nIf a baker has 10 loaves and sells 3, then bakes 5 more, how many does he have?<|im_end|>\n<|im_start|>assistant\n"
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, eos_token_id = tokenizer.eos_token_id, tokenizer = tokenizer, streamer = text_streamer, max_new_tokens = 512, stop_strings=["<|im_end|>"])

### 7. Save Model
Mount Google Drive to save the LoRA adapters permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained("/content/drive/MyDrive/scot-qwen-adapters")
tokenizer.save_pretrained("/content/drive/MyDrive/scot-qwen-adapters")